# Results analysis — EEG-only vs. EEG+ECG

Reads the `metrics.csv` logs produced by `evaluate_eeg.py` (one row per run/seed),
aggregates **mean ± std** per feature set, and produces the comparison figure for the paper.

Run several seeds first, e.g.:
```
python train_model_eeg.py --epochs 30 --random-state 42 ; python evaluate_eeg.py
python train_model_eeg.py --epochs 30 --random-state 43 ; python evaluate_eeg.py
```
Each `evaluate_eeg.py` run appends a row. EEG+ECG bars appear automatically once that
pipeline writes its own `metrics.csv` (with `feature_set=eeg_ecg`).

In [ ]:
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# One metrics.csv per pipeline; each evaluate run appends a row.
METRICS_FILES = [
    Path('../results/seizure_detection_eeg/metrics.csv'),
    Path('../results/seizure_detection_eeg_ecg/metrics.csv'),  # appears later
]
OUT_PNG = Path('../results/comparison_eeg_vs_eeg_ecg.png')
METRICS = ['precision', 'recall', 'f1']

In [ ]:
# Load all runs from whichever metrics files exist.
rows = []
for f in METRICS_FILES:
    if f.exists():
        with open(f, newline='', encoding='utf-8') as fh:
            rows.extend(list(csv.DictReader(fh)))
print(f'Loaded {len(rows)} run(s) from {sum(f.exists() for f in METRICS_FILES)} file(s).')
rows[:2]

In [ ]:
# Aggregate mean +/- std of each metric, per feature set.
def aggregate(rows, feature_set):
    out = {}
    for m in METRICS:
        vals = [float(r[m]) for r in rows if r.get('feature_set') == feature_set]
        if vals:
            out[m] = (float(np.mean(vals)), float(np.std(vals)), len(vals))
    return out

groups = sorted({r['feature_set'] for r in rows})
summary = {g: aggregate(rows, g) for g in groups}
for g, s in summary.items():
    print(g, {m: f'{mu:.3f}±{sd:.3f} (n={n})' for m, (mu, sd, n) in s.items()})

In [ ]:
# Grouped bar chart with error bars: EEG-only vs EEG+ECG.
order = [g for g in ['eeg', 'eeg_ecg'] if g in summary] or groups
x = np.arange(len(METRICS))
width = 0.8 / max(1, len(order))

fig, ax = plt.subplots(figsize=(8, 5))
for i, g in enumerate(order):
    means = [summary[g][m][0] for m in METRICS]
    stds = [summary[g][m][1] for m in METRICS]
    ax.bar(x + i * width, means, width, yerr=stds, capsize=4,
           label=g.upper().replace('_', '+'))

ax.set_xticks(x + width * (len(order) - 1) / 2)
ax.set_xticklabels([m.capitalize() for m in METRICS])
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('EEG-only vs. EEG+ECG  (mean ± std over runs)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=150)
print('Saved', OUT_PNG.resolve())
plt.show()

In [ ]:
# Summary table (paste-ready numbers for the paper).
header = f"{'feature_set':<12}{'precision':>16}{'recall':>16}{'f1':>16}"
print(header)
print('-' * len(header))
for g in order:
    cells = [g]
    for m in METRICS:
        if m in summary[g]:
            mu, sd, n = summary[g][m]
            cells.append(f'{mu:.3f}±{sd:.3f}')
        else:
            cells.append('-')
    print(f'{cells[0]:<12}{cells[1]:>16}{cells[2]:>16}{cells[3]:>16}')